In [20]:
"""
XRD Voigt peak fitting for κ/β/α-Ga₂O₃ epitaxial films
Paper: Deciphering three‑phase competition of Ga₂O₃ on c‑plane sapphire via XRD‑assisted machine learning
Journal: Crystal Growth & Design (CGD)
Author: Zhao Zhang
Date: 2026
Description:
    This script loads raw XRD txt data, performs multi-peak Voigt fitting,
    calculates phase fraction, crystallite size (Scherrer equation), microstrain,
    exports fitted figures (1200 dpi PNG + vector PDF) and summary table (CSV).
Raw XRD data: txt files stored in folder "xrd_data/"
Output: Figures and CSV summary table saved in folder "DATA/"

Dependencies:
    numpy, pandas, matplotlib, lmfit, scipy
> requirements.txt
numpy==1.24.3
pandas==2.0.2
matplotlib==3.7.1
lmfit==1.2.1
scipy==1.10.1
scikit-learn==1.2.2
License: MIT
"""
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pathlib
import warnings
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
from lmfit import Model
from scipy.stats import pearsonr
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# ===================== CGD Journal Style: Times New Roman, English Only =====================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.formatter.useoffset'] = False

# Font size & bold for journal figures
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# XRD physical constants
XRD_LAMBDA = 1.5406    # Cu Kα wavelength (Å)
SCHERRER_K = 0.9
FWHM_INST = 0.05       # Instrumental broadening (°)
MIN_GOOD_SAMPLES = 3

# Folder definitions
PLOT_FOLDER = "DATA"
os.makedirs(PLOT_FOLDER, exist_ok=True)

# Range definition
SUB_MASK_RANGE = (41.5, 42.0)
FIT_RANGE = (37.0, 41.0)

# ===================== Phase peak positions (2θ, degree) =====================
_ALPHA_PEAKS = np.array([39.9, 40.2])
_BETA_PEAKS  = np.array([38.388])
_KAPPA_PEAKS = np.array([38.769])
_ALL_PEAKS_FULL = sorted(np.concatenate([_ALPHA_PEAKS, _BETA_PEAKS, _KAPPA_PEAKS]))
ALL_PEAKS = np.array([p for p in _ALL_PEAKS_FULL if FIT_RANGE[0] <= p <= FIT_RANGE[1]])

def label_phase(peak):
    """Map peak position to phase index: 0=α,1=β,2=κ"""
    if peak in _ALPHA_PEAKS:
        return 0
    elif peak in _BETA_PEAKS:
        return 1
    else:
        return 2

PHASE_OF_PEAK = [label_phase(p) for p in ALL_PEAKS]
ALPHA_IDX = [i for i, ph in enumerate(PHASE_OF_PEAK) if ph == 0]
BETA_IDX  = [i for i, ph in enumerate(PHASE_OF_PEAK) if ph == 1]
KAPPA_IDX = [i for i, ph in enumerate(PHASE_OF_PEAK) if ph == 2]

def get_peak_bounds(peak, phase):
    """Define fitting boundary for each peak according to phase"""
    if phase == 0:
        return (peak - 0.20, peak + 0.20)
    elif phase == 1:
        return (peak - 0.25, peak + 0.25)
    else:
        return (peak - 0.50, peak + 0.50)

PEAK_BOUNDS = [get_peak_bounds(ALL_PEAKS[i], PHASE_OF_PEAK[i]) for i in range(len(ALL_PEAKS))]


In [6]:
# ===================== Basic physical & fitting functions =====================
def voigt_func(x, amp, center, sigma, gamma):
    """Voigt profile: convolution of Gaussian + Lorentzian"""
    g = np.exp(-(x - center)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))
    l = gamma / (np.pi * ((x - center)**2 + gamma**2))
    return amp * (0.5 * g + 0.5 * l)

def calc_voigt_fwhm(sigma, gamma):
    """Calculate FWHM from Voigt parameters"""
    fwhm_g = 2 * sigma * np.sqrt(2 * np.log(2))
    fwhm_l = 2 * gamma
    return 0.5346 * fwhm_l + np.sqrt(0.2166 * fwhm_l**2 + fwhm_g**2)

def subtract_background(x, y, poly_order=3, iterations=5):
    """Iterative polynomial background subtraction"""
    y_ = y.copy()
    for _ in range(iterations):
        p = np.polyfit(x, y_, poly_order)
        bg = np.polyval(p, x)
        y_ = np.minimum(y_, bg)
    p = np.polyfit(x, y_, poly_order)
    bg = np.polyval(p, x)
    return y - bg, bg

def scherrer_size(fwhm, two_theta):
    """Scherrer equation: calculate crystallite size (nm)"""
    fwhm_corr = np.sqrt(max(fwhm**2 - FWHM_INST**2, 0.0001))
    theta_rad = np.radians(two_theta / 2)
    D_ang = (SCHERRER_K * XRD_LAMBDA) / (np.radians(fwhm_corr) * np.cos(theta_rad))
    return D_ang / 10

def calc_microstrain(fwhm, two_theta):
    """Calculate microstrain from peak broadening"""
    fwhm_corr = np.sqrt(max(fwhm**2 - FWHM_INST**2, 0.0001))
    theta_rad = np.radians(two_theta / 2)
    return np.radians(fwhm_corr) / (4 * np.tan(theta_rad))

def load_xrd_data(folder, pattern):
    """Load all XRD txt files, align theta axis"""
    intensity_list, params, thetas = [], [], []
    for f in os.listdir(folder):
        if f.endswith(".txt"):
            match = re.match(pattern, f)
            if match:
                ar, p, si, t = int(match[1]), int(match[2]), int(match[3]), int(match[4])
                params.append([ar, p, si, t])
                data = np.loadtxt(os.path.join(folder, f), delimiter=',', comments='#')
                thetas.append(data[:, 0])
                intensity_list.append(data[:, 1])
    x_ref = thetas[0]
    aligned = []
    for th, i in zip(thetas, intensity_list):
        if not np.allclose(th, x_ref):
            aligned.append(np.interp(x_ref, th, i))
        else:
            aligned.append(i)
    return x_ref, aligned, np.array(params)

def time_to_thickness_group(time_val):
    """Classify sample by growth time (thin/thick film)"""
    return "Thin film (~100 nm)" if time_val <= 4000 else "Thick film (~400 nm)"


In [7]:
# ===================== Fitting functions =====================
def fit_free_no_bg(x, y):
    try:
        model = None
        for i, cen in enumerate(ALL_PEAKS):
            m = Model(voigt_func, prefix=f"p{i}_")
            model = m if model is None else model + m
        model += Model(lambda x, b0, b1: b0 + b1*x, prefix="bg_")
        params = model.make_params()
        for i, cen in enumerate(ALL_PEAKS):
            low, high = PEAK_BOUNDS[i]
            params.add(f"p{i}_amp", value=np.max(y)*0.2, min=0)
            params.add(f"p{i}_center", value=cen, min=low, max=high)
            params.add(f"p{i}_sigma", value=0.1, min=0.03, max=0.5)
            params.add(f"p{i}_gamma", value=0.1, min=0.03, max=0.5)
        params.add("bg_b0", value=np.min(y), min=0)
        params.add("bg_b1", value=0, min=-0.1, max=0.1)
        res = model.fit(y, params, x=x, method='leastsq', nan_policy='omit')
        fit_y = res.best_fit
        rwp = np.sqrt(np.sum((y-fit_y)**2)/np.sum(y**2))*100
        peak_fwhms = [calc_voigt_fwhm(res.params[f"p{i}_sigma"].value, res.params[f"p{i}_gamma"].value) for i in range(len(ALL_PEAKS))]
        def phase_params(idx_list):
            if not idx_list:
                return 0.0, np.nan, np.nan, 0.0
            peaks = [res.params[f"p{ii}_center"].value for ii in idx_list]
            fwhms = [peak_fwhms[ii] for ii in idx_list]
            amps = [res.params[f"p{ii}_amp"].value for ii in idx_list]
            total_amp = sum(amps)
            if total_amp <= 0:
                return 0.0, np.nan, np.nan, 0.0
            main_idx = np.argmax(amps)
            D = scherrer_size(fwhms[main_idx], peaks[main_idx])
            E = calc_microstrain(fwhms[main_idx], peaks[main_idx])
            return total_amp, D, E, total_amp
        aa, da, ea, _ = phase_params(ALPHA_IDX)
        ab, db, eb, _ = phase_params(BETA_IDX)
        ak, dk, ek, _ = phase_params(KAPPA_IDX)
        tot = aa + ab + ak + 1e-8
        return {
            "fit_y": fit_y, "RWP": rwp,
            "ratio_alpha": aa/tot, "ratio_beta": ab/tot, "ratio_kappa": ak/tot,
            "D_alpha": da, "E_alpha": ea, "D_beta": db, "E_beta": eb, "D_kappa": dk, "E_kappa": ek,
            "amp_alpha": aa, "amp_beta": ab, "amp_kappa": ak, "params": res.params,
            "sigma": [res.params[f"p{i}_sigma"].value for i in range(len(ALL_PEAKS))],
            "gamma": [res.params[f"p{i}_gamma"].value for i in range(len(ALL_PEAKS))]
        }
    except Exception as e:
        warnings.warn(f"Fit no-bg failed: {str(e)}")
        return None

def fit_free_with_bg(x, y):
    try:
        y_sub, bg = subtract_background(x, y)
        if np.max(y_sub) < 1e-3:
            return None
        model = None
        for i, cen in enumerate(ALL_PEAKS):
            m = Model(voigt_func, prefix=f"p{i}_")
            model = m if model is None else model + m
        params = model.make_params()
        for i, cen in enumerate(ALL_PEAKS):
            low, high = PEAK_BOUNDS[i]
            params.add(f"p{i}_amp", value=np.max(y_sub)*0.2, min=0)
            params.add(f"p{i}_center", value=cen, min=low, max=high)
            params.add(f"p{i}_sigma", value=0.1, min=0.03, max=0.5)
            params.add(f"p{i}_gamma", value=0.1, min=0.03, max=0.5)
        res = model.fit(y_sub, params, x=x, method='leastsq', nan_policy='omit')
        fit_y = res.best_fit + bg
        rwp = np.sqrt(np.sum((y-fit_y)**2)/np.sum(y**2))*100
        peak_fwhms = [calc_voigt_fwhm(res.params[f"p{i}_sigma"].value, res.params[f"p{i}_gamma"].value) for i in range(len(ALL_PEAKS))]
        def phase_params(idx_list):
            if not idx_list:
                return 0.0, np.nan, np.nan, 0.0
            peaks = [res.params[f"p{ii}_center"].value for ii in idx_list]
            fwhms = [peak_fwhms[ii] for ii in idx_list]
            amps = [res.params[f"p{ii}_amp"].value for ii in idx_list]
            total_amp = sum(amps)
            if total_amp <= 0:
                return 0.0, np.nan, np.nan, 0.0
            main_idx = np.argmax(amps)
            D = scherrer_size(fwhms[main_idx], peaks[main_idx])
            E = calc_microstrain(fwhms[main_idx], peaks[main_idx])
            return total_amp, D, E, total_amp
        aa, da, ea, _ = phase_params(ALPHA_IDX)
        ab, db, eb, _ = phase_params(BETA_IDX)
        ak, dk, ek, _ = phase_params(KAPPA_IDX)
        tot = aa + ab + ak + 1e-8
        return {
            "fit_y": fit_y, "RWP": rwp,
            "ratio_alpha": aa/tot, "ratio_beta": ab/tot, "ratio_kappa": ak/tot,
            "D_alpha": da, "E_alpha": ea, "D_beta": db, "E_beta": eb, "D_kappa": dk, "E_kappa": ek,
            "amp_alpha": aa, "amp_beta": ab, "amp_kappa": ak, "params": res.params,
            "sigma": [res.params[f"p{i}_sigma"].value for i in range(len(ALL_PEAKS))],
            "gamma": [res.params[f"p{i}_gamma"].value for i in range(len(ALL_PEAKS))]
        }
    except Exception as e:
        warnings.warn(f"Fit with-bg failed: {str(e)}")
        return None

def fit_with_good_init(x, y, good_sigma, good_gamma):
    try:
        model = None
        for i, cen in enumerate(ALL_PEAKS):
            m = Model(voigt_func, prefix=f"p{i}_")
            model = m if model is None else model + m
        model += Model(lambda x, b0, b1: b0 + b1*x, prefix="bg_")
        params = model.make_params()
        for i, cen in enumerate(ALL_PEAKS):
            low, high = PEAK_BOUNDS[i]
            params.add(f"p{i}_amp", value=np.max(y)*0.2, min=0)
            params.add(f"p{i}_center", value=cen, min=low, max=high)
            params.add(f"p{i}_sigma", value=good_sigma[i], min=0.03, max=0.8)
            params.add(f"p{i}_gamma", value=good_gamma[i], min=0.03, max=0.8)
        params.add("bg_b0", value=np.min(y), min=0)
        params.add("bg_b1", value=0, min=-0.1, max=0.1)
        res = model.fit(y, params, x=x, method='leastsq', nan_policy='omit')
        fit_y = res.best_fit
        rwp = np.sqrt(np.sum((y-fit_y)**2)/np.sum(y**2))*100
        aa = sum(res.params[f"p{i}_amp"].value for i in ALPHA_IDX)
        ab = sum(res.params[f"p{i}_amp"].value for i in BETA_IDX)
        ak = sum(res.params[f"p{i}_amp"].value for i in KAPPA_IDX)
        tot = aa + ab + ak + 1e-8
        return {
            "fit_y": fit_y, "RWP": rwp,
            "ratio_alpha": aa/tot, "ratio_beta": ab/tot, "ratio_kappa": ak/tot,
            "amp_alpha": aa, "amp_beta": ab, "amp_kappa": ak
        }
    except Exception as e:
        warnings.warn(f"Guided fit failed: {str(e)}")
        return None


In [8]:
# ===================== Plot & Export functions =====================
def save_fig(filename):
    """
    Export figure for journal submission:
        PNG: 1200 dpi (for manuscript review)
        PDF: vector graphic (for final publication)
    """
    png_path = os.path.join(PLOT_FOLDER, f"{filename}.png")
    pdf_path = os.path.join(PLOT_FOLDER, f"{filename}.pdf")
    plt.savefig(png_path, dpi=1200, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.close()

def plot_fit_only(x, y, final_res, idx, param):
    plt.figure(figsize=(7, 4.5))
    plt.plot(x, y, 'k-', label="Raw XRD", lw=1.2)
    plt.plot(x, final_res["fit_y"], 'r-', lw=2.0, label="Fitting")
    plt.title(f"Sample {idx+1} | Ar={param[0]}, P={param[1]} mbar, Si={param[2]}, T={param[3]} s")
    plt.xlabel("2θ (°)")
    plt.ylabel("Intensity (a.u.)")
    plt.legend(loc='upper left')
    plt.grid(alpha=0.3, linestyle='--')
    save_fig(f"Sample_{idx+1}")

def plot_sample9_10_compare(x_full, y_list_full):
    idx9 = 8
    idx10 = 9
    mask = (x_full >= FIT_RANGE[0]) & (x_full <= FIT_RANGE[1])
    x = x_full[mask]
    y9 = np.asarray(y_list_full[idx9])[mask]
    y10 = np.asarray(y_list_full[idx10])[mask]
    plt.figure(figsize=(7, 4.5))
    plt.plot(x, y9, 'k-', label="Sample 9", lw=1.2)
    plt.plot(x, y10, 'r-', label="Sample 10", lw=1.2)
    plt.xlabel("2θ (°)")
    plt.ylabel("Intensity (a.u.)")
    plt.title("XRD comparison of Sample 9 and Sample 10 (37°–41°)")
    plt.legend(loc='upper left')
    plt.grid(alpha=0.3, linestyle='--')
    save_fig("Sample9_10_compare")


In [9]:
# ===================== Main workflow =====================
if __name__ == "__main__":
    DATA_FOLDER = "xrd_data"
    FILE_REGEX = r"TE_Ar(\d+)_P(\d+)mbar_Si(\d+)_T(\d+)s\.txt"
    print("🔍 Loading XRD raw data...")
    x_full, y_list_full, params = load_xrd_data(DATA_FOLDER, FILE_REGEX)
    print(f"✅ Total {len(y_list_full)} samples loaded")

    mask = (x_full >= FIT_RANGE[0]) & (x_full <= FIT_RANGE[1])
    x, y_list = x_full[mask], [y[mask] for y in y_list_full]
    # Fixed bug: original sub_mask logic error
    sub_mask = (x_full >= SUB_MASK_RANGE[0]) & (x_full <= SUB_MASK_RANGE[1])

    sample_info, all_best_init = [], []
    print("\n========== First round fitting ==========")
    for idx, (y, p) in enumerate(zip(y_list, params)):
        res_nobg = fit_free_no_bg(x, y)
        res_wbg = fit_free_with_bg(x, y)
        rwp2 = res_wbg["RWP"] if res_wbg else 999
        if res_nobg is None and res_wbg is None:
            warnings.warn(f"Sample {idx+1}: both fitting failed")
            sample_info.append({"nobg_res": None, "wbg_res": None, "best_init": None, "p": p})
            all_best_init.append(None)
            continue
        elif res_nobg is None:
            best_init = res_wbg
        elif res_wbg is None:
            best_init = res_nobg
        else:
            best_init = res_nobg if res_nobg["RWP"] <= rwp2 else res_wbg
        sample_info.append({"nobg_res": res_nobg, "wbg_res": res_wbg, "best_init": best_init, "p": p})
        all_best_init.append(best_init)
        print(f"Sample {idx+1} best RWP: {best_init['RWP']:.2f}%")

    valid_samples = [s for s in all_best_init if s is not None]
    if len(valid_samples) >= MIN_GOOD_SAMPLES:
        good_samples = sorted(valid_samples, key=lambda x: x["RWP"])[:MIN_GOOD_SAMPLES]
        good_sigma = np.mean([s["sigma"] for s in good_samples], axis=0)
        good_gamma = np.mean([s["gamma"] for s in good_samples], axis=0)
        print(f"\n✅ Good-sample peak shape parameters loaded")
    else:
        raise RuntimeError("Not enough valid fitted samples for initial guess calibration")

    final_result_list = []
    print("\n========== Final guided fitting ==========")
    for idx, info in enumerate(sample_info):
        y_raw, p = y_list[idx], info["p"]
        init_best = info["best_init"]
        if init_best is None:
            final_result_list.append({
                "Sample": idx + 1,
                "Ar(sccm)": p[0],
                "P(mbar)": p[1],
                "Si(μmol/min)": p[2],
                "Epitaxial growth time(s)": p[3],
                "Thickness groups": time_to_thickness_group(p[3]),
                "Final RWP": np.nan,
                "α-phase fraction": np.nan,
                "β-phase fraction": np.nan,
                "κ-phase fraction": np.nan,
                "β-phase grain size (nm)": np.nan,
                "β-phase microstrain": np.nan,
                "κ-phase grain size (nm)": np.nan,
                "κ-phase microstrain": np.nan,
                "Status": "Fit Failed"
            })
            continue
        guided_res = fit_with_good_init(x, y_raw, good_sigma, good_gamma)
        final_res = guided_res if (guided_res is not None and guided_res["RWP"] < init_best["RWP"]) else init_best
        final_Db, final_Eb = init_best["D_beta"], init_best["E_beta"]
        final_Dk, final_Ek = init_best["D_kappa"], init_best["E_kappa"]
        group = time_to_thickness_group(p[3])
        plot_fit_only(x, y_raw, final_res, idx, p)

        ratio_alpha = round(final_res["ratio_alpha"], 3)
        ratio_beta = round(final_res["ratio_beta"], 3)
        ratio_kappa = round(final_res["ratio_kappa"], 3)
        if ratio_beta <= 0:
            final_Db, final_Eb = np.nan, np.nan
        if ratio_kappa <= 0:
            final_Dk, final_Ek = np.nan, np.nan

        final_result_list.append({
            "Sample": idx + 1,
            "Ar(sccm)": p[0],
            "P(mbar)": p[1],
            "Si(μmol/min)": p[2],
            "Epitaxial growth time(s)": p[3],
            "Thickness groups": group,
            "Final RWP": round(final_res["RWP"], 2),
            "α-phase fraction": ratio_alpha,
            "β-phase fraction": ratio_beta,
            "κ-phase fraction": ratio_kappa,
            "β-phase grain size (nm)": round(final_Db, 1) if not np.isnan(final_Db) else np.nan,
            "β-phase microstrain": round(final_Eb, 6) if not np.isnan(final_Eb) else np.nan,
            "κ-phase grain size (nm)": round(final_Dk, 1) if not np.isnan(final_Dk) else np.nan,
            "κ-phase microstrain": round(final_Ek, 6) if not np.isnan(final_Ek) else np.nan,
            "Status": "Success"
        })
    df = pd.DataFrame(final_result_list)
    csv_path = os.path.join(PLOT_FOLDER, "XRD_final_results.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV summary table saved: {csv_path}")

    plot_sample9_10_compare(x_full, y_list_full)
    print("✅ Sample 9 & Sample 10 comparison plot saved")
    print("\n🎉 All calculation and plotting finished.")


🔍 Loading XRD raw data...
✅ Total 16 samples loaded

========== First round fitting ==========
Sample 1 best RWP: 7.81%
Sample 2 best RWP: 5.81%
Sample 3 best RWP: 6.15%
Sample 4 best RWP: 5.79%
Sample 5 best RWP: 6.91%
Sample 6 best RWP: 8.39%
Sample 7 best RWP: 4.69%
Sample 8 best RWP: 6.82%
Sample 9 best RWP: 8.86%
Sample 10 best RWP: 7.99%
Sample 11 best RWP: 11.87%
Sample 12 best RWP: 4.36%
Sample 13 best RWP: 19.75%
Sample 14 best RWP: 8.24%
Sample 15 best RWP: 14.97%
Sample 16 best RWP: 12.47%

✅ Good-sample peak shape parameters loaded

========== Final guided fitting ==========

✅ CSV summary table saved: DATA\XRD_final_results.csv
✅ Sample 9 & Sample 10 comparison plot saved

🎉 All calculation and plotting finished.


In [21]:
# =============================================================================
# Plot Fig3 main figure and TOC graphic for journal submission
# Data source: XRD_final_results.csv stored in ./DATA
# Output: PNG (1200 dpi) + vector PDF saved into ./DATA folder
# Font style, size and bold settings consistent with phase fraction plot
# =============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -------------------------- Journal Figure Style Settings --------------------------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 1200
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.linewidth'] = 1.0

# Font size & bold configuration, fully consistent with phase fraction figure
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16

# -------------------------- Relative File Path Configuration --------------------------
DATA_FOLDER = "DATA"
os.makedirs(DATA_FOLDER, exist_ok=True)
CSV_PATH = os.path.join(DATA_FOLDER, "XRD_final_results.csv")

# Load and clean dataset
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
df['Thickness groups'] = df['Thickness groups'].astype(str).str.strip()

# Axis label definitions and x coordinates
x_full_labels = ["20 mbar-40 sccm", "20 mbar-60 sccm", "20 mbar-80 sccm", "40 mbar-80 sccm"]
x_toc_labels  = ["20-40", "20-60", "20-80", "40-80"]
x_positions = np.array([0, 1.6, 3.2, 4.8])
y_lim_plot = [-0.08, 0.42]

# Phase color mapping
phase_color = {
    'α': '#c0392b',
    'β': '#27ae60',
    'κ': '#2980b9'
}

# Threshold for phase identification
PURE_THRESHOLD = 0.90
EXIST_THRESHOLD = 0.10

# -------------------------- Phase judgement helper functions --------------------------
def filter_phases(fa, fb, fk):
    """Filter phases with fraction above existence threshold"""
    phases = {'α': fa, 'β': fb, 'κ': fk}
    valid = {k: v for k, v in phases.items() if v >= EXIST_THRESHOLD}
    return valid

def is_pure_phase(fa, fb, fk):
    """Judge whether the sample is single-phase dominated"""
    return max(fa, fb, fk) >= PURE_THRESHOLD

def get_dominant_phase(fa, fb, fk):
    """Return the phase with maximum fraction"""
    frac = {'α': fa, 'β': fb, 'κ': fk}
    return max(frac, key=frac.get)

def get_phase_label(fa, fb, fk):
    """Generate combined label for mixed phases"""
    valid = filter_phases(fa, fb, fk)
    order = ['κ', 'α', 'β']
    sorted_phases = [p for p in order if p in valid]
    return '+'.join(sorted_phases)

def mix_phase_color(fa, fb, fk):
    """Calculate blended RGB color for mixed-phase samples"""
    valid = filter_phases(fa, fb, fk)
    total = sum(valid.values())
    r, g, b = 0.12, 0.12, 0.12
    if 'α' in valid:
        r += (valid['α'] / total) * 0.82
    if 'β' in valid:
        g += (valid['β'] / total) * 0.82
    if 'κ' in valid:
        b += (valid['κ'] / total) * 0.82
    r = min(max(r, 0), 1)
    g = min(max(g, 0), 1)
    b = min(max(b, 0), 1)
    return (r, g, b)

# -------------------------- Main Figure (Fig3) --------------------------
def plot_main_optimized():
    fig, ax = plt.subplots(figsize=(14, 6.8))
    all_points = []
    pair_dict = {}

    for x_idx, cond_label in enumerate(x_full_labels):
        p_str, ar_str = cond_label.split("-")
        p_val = float(p_str.replace(" mbar", ""))
        ar_val = float(ar_str.replace(" sccm", ""))
        sub_df = df[np.isclose(df['P(mbar)'], p_val) & np.isclose(df['Ar(sccm)'], ar_val)]

        for _, row in sub_df.iterrows():
            si_raw = row['Si(μmol/min)']
            y_plot = si_raw * 100
            fa = row['α-phase fraction']
            fb = row['β-phase fraction']
            fk = row['κ-phase fraction']
            dominant = get_dominant_phase(fa, fb, fk)
            pure = is_pure_phase(fa, fb, fk)
            thick = row['Thickness groups']
            halo_color = None
            if not pure:
                halo_color = mix_phase_color(fa, fb, fk)

            if "Thin film" in thick:
                x_plot = x_positions[x_idx] - 0.20
            else:
                x_plot = x_positions[x_idx] + 0.20

            point_info = {
                'x': x_plot,
                'y': y_plot,
                'dominant': dominant,
                'thick': thick,
                'pure': pure,
                'halo_color': halo_color
            }
            all_points.append(point_info)

            pair_key = (x_idx, round(si_raw, 6))
            if pair_key not in pair_dict:
                pair_dict[pair_key] = {'thin': None, 'thick': None}
            if "Thin film" in thick:
                pair_dict[pair_key]['thin'] = (x_plot, y_plot)
            else:
                pair_dict[pair_key]['thick'] = (x_plot, y_plot)

    # Plot mixed-phase halo background
    for pt in all_points:
        if not pt['pure'] and pt['halo_color'] is not None:
            ax.scatter(
                pt['x'], pt['y'],
                s=500,
                color=pt['halo_color'],
                alpha=0.45,
                marker='o',
                zorder=2
            )

    # Draw arrows from thin film to thick film
    for pair_key, pos in pair_dict.items():
        p_thin = pos['thin']
        p_thick = pos['thick']
        if p_thin is not None and p_thick is not None:
            ax.annotate(
                '', xy=p_thick, xytext=p_thin,
                arrowprops=dict(arrowstyle='->', color='#444444', lw=1.2),
                zorder=3
            )

    # Plot data markers
    for pt in all_points:
        c = phase_color[pt['dominant']]
        if "Thin film" in pt['thick']:
            ax.scatter(
                pt['x'], pt['y'],
                marker='o', facecolor='white', edgecolor=c,
                s=110, linewidth=1.8, zorder=5
            )
        else:
            ax.scatter(
                pt['x'], pt['y'],
                marker='s', facecolor=c, edgecolor='black',
                s=110, linewidth=1.0, zorder=5
            )

    ax.axhline(y=0, color='black', linewidth=0.9, linestyle='--', zorder=1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_full_labels, rotation=0, ha='center')
    ax.set_xlabel("Growth condition (Vapor-phase supersaturation)", labelpad=12)
    ax.set_ylabel(r"Si doping flux ($\times10^{-2}\ \mu\mathrm{mol/min}$)", labelpad=12)
    ax.set_ylim(y_lim_plot)
    ax.set_xlim(-0.7, x_positions[-1] + 1.0)
    ax.grid(axis='y', alpha=0.3, linestyle='--', zorder=0)

    # Bold tick labels
    ax.tick_params(axis='both', which='major', labelsize=17)
    for label in ax.get_xticklabels():
        label.set_weight('bold')
    for label in ax.get_yticklabels():
        label.set_weight('bold')

    # Custom legend
    legend_handles = [
        Line2D([], [], marker='o', color='w', markerfacecolor='white', markeredgecolor='black',
               markersize=10, linewidth=0, label='Thin film (~100 nm)'),
        Line2D([], [], marker='s', color='w', markerfacecolor='black', markeredgecolor='black',
               markersize=10, linewidth=0, label='Thick film (~400 nm)'),
        Line2D([], [], marker='', color='#444444', lw=1.2, label='Thin → Thick'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#c0392b', markeredgecolor='black',
               markersize=10, linewidth=0, label=r'$\alpha$-dominant'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#27ae60', markeredgecolor='black',
               markersize=10, linewidth=0, label=r'$\beta$-dominant'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#2980b9', markeredgecolor='black',
               markersize=10, linewidth=0, label=r'$\kappa$-dominant'),
        Line2D([], [], marker='o', color='w', markerfacecolor='gray', markeredgecolor='gray',
               markersize=14, alpha=0.45, linewidth=0, label='Mixed phase region'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', frameon=False, bbox_to_anchor=(1, 1),
              prop={'weight':'bold'})

    plt.tight_layout()
    save_png = os.path.join(DATA_FOLDER, "Fig3_main_optimized.png")
    save_pdf = os.path.join(DATA_FOLDER, "Fig3_main_optimized.pdf")
    plt.savefig(save_png, bbox_inches='tight')
    plt.savefig(save_pdf, bbox_inches='tight')
    plt.close()
    print(f"✅ Main figure (Fig3) saved\n{save_png}\n{save_pdf}")

# -------------------------- TOC Graph --------------------------
def plot_toc_simplified():
    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    all_points = []
    pair_dict = {}

    for x_idx, cond_label in enumerate(x_full_labels):
        p_str, ar_str = cond_label.split("-")
        p_val = float(p_str.replace(" mbar", ""))
        ar_val = float(ar_str.replace(" sccm", ""))
        sub_df = df[np.isclose(df['P(mbar)'], p_val) & np.isclose(df['Ar(sccm)'], ar_val)]

        for _, row in sub_df.iterrows():
            si_raw = row['Si(μmol/min)']
            y_plot = si_raw * 100
            fa = row['α-phase fraction']
            fb = row['β-phase fraction']
            fk = row['κ-phase fraction']
            dominant = get_dominant_phase(fa, fb, fk)
            pure = is_pure_phase(fa, fb, fk)
            thick = row['Thickness groups']
            halo_color = None
            if not pure:
                halo_color = mix_phase_color(fa, fb, fk)

            if "Thin film" in thick:
                x_plot = x_positions[x_idx] - 0.20
            else:
                x_plot = x_positions[x_idx] + 0.20

            all_points.append({
                'x': x_plot,
                'y': y_plot,
                'dominant': dominant,
                'thick': thick,
                'pure': pure,
                'halo_color': halo_color
            })
            pair_key = (x_idx, round(si_raw, 6))
            if pair_key not in pair_dict:
                pair_dict[pair_key] = {'thin': None, 'thick': None}
            if "Thin film" in thick:
                pair_dict[pair_key]['thin'] = (x_plot, y_plot)
            else:
                pair_dict[pair_key]['thick'] = (x_plot, y_plot)

    # Mixed phase halo
    for pt in all_points:
        if not pt['pure'] and pt['halo_color'] is not None:
            ax.scatter(pt['x'], pt['y'], s=420, color=pt['halo_color'], alpha=0.45, marker='o', zorder=2)

    # Thin to thick arrows
    for pair_key, pos in pair_dict.items():
        p_thin = pos['thin']
        p_thick = pos['thick']
        if p_thin is not None and p_thick is not None:
            ax.annotate('', xy=p_thick, xytext=p_thin,
                        arrowprops=dict(arrowstyle='->', color='#444444', lw=1.1), zorder=3)

    # Plot markers
    for pt in all_points:
        c = phase_color[pt['dominant']]
        if "Thin film" in pt['thick']:
            ax.scatter(pt['x'], pt['y'], marker='o', facecolor='white', edgecolor=c,
                       s=95, linewidth=1.7, zorder=5)
        else:
            ax.scatter(pt['x'], pt['y'], marker='s', facecolor=c, edgecolor='black',
                       s=95, linewidth=0.9, zorder=5)

    ax.axhline(y=0, color='black', linewidth=0.9, linestyle='--', zorder=1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_toc_labels, rotation=0, ha='center')
    ax.set_xlabel("Growth condition", labelpad=12)
    ax.set_ylabel(r"Si-flux ($\times10^{-2}\ \mu\mathrm{mol/min}$)", labelpad=12)
    ax.set_ylim(y_lim_plot)
    ax.set_xlim(-0.7, x_positions[-1] + 0.7)
    ax.grid(axis='y', alpha=0.3, linestyle='--', zorder=0)

    # Bold tick labels
    ax.tick_params(axis='both', which='major', labelsize=17)
    for label in ax.get_xticklabels():
        label.set_weight('bold')
    for label in ax.get_yticklabels():
        label.set_weight('bold')

    # TOC legend
    legend_handles = [
        Line2D([], [], marker='o', color='w', markerfacecolor='white', markeredgecolor='black',
               markersize=9, linewidth=0, label='Thin film'),
        Line2D([], [], marker='s', color='w', markerfacecolor='black', markeredgecolor='black',
               markersize=9, linewidth=0, label='Thick film'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#c0392b', markeredgecolor='black',
               markersize=9, linewidth=0, label=r'$\alpha$'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#27ae60', markeredgecolor='black',
               markersize=9, linewidth=0, label=r'$\beta$'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#2980b9', markeredgecolor='black',
               markersize=9, linewidth=0, label=r'$\kappa$'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', frameon=False, prop={'weight':'bold'})

    plt.tight_layout()
    save_png = os.path.join(DATA_FOLDER, "Fig3_TOC_simplified.png")
    save_pdf = os.path.join(DATA_FOLDER, "Fig3_TOC_simplified.pdf")
    plt.savefig(save_png, bbox_inches='tight')
    plt.savefig(save_pdf, bbox_inches='tight')
    plt.close()
    print(f"✅ TOC graphic saved\n{save_png}\n{save_pdf}")

if __name__ == "__main__":
    plot_main_optimized()
    plot_toc_simplified()
    print("\n🎉 All figures generated successfully")


✅ Main figure (Fig3) saved
DATA\Fig3_main_optimized.png
DATA\Fig3_main_optimized.pdf
✅ TOC graphic saved
DATA\Fig3_TOC_simplified.png
DATA\Fig3_TOC_simplified.pdf

🎉 All figures generated successfully


In [14]:
# ============================================================
# Generate two publication-quality figures from XRD_final_results.csv
# Fig1: β/κ grain size vs microstrain linear fitting
# Fig2: Phase fraction of three phases under different growth conditions (thin/thick film comparison)
# All input/output files inside DATA folder
# Font settings match Fig3: labelsize=18, tick=17, legend=16, title=18; all text bold
# ============================================================

# ---------- Journal global style (consistent with Fig3) ----------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.formatter.useoffset'] = False
# ========= Font size & bold setting, consistent with Fig3 =========
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.titlesize'] = 18
# Bold for all text
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 0.8

# ---------- Pure relative path: all data & figures stored in DATA ----------
DATA_FOLDER = "./DATA"
os.makedirs(DATA_FOLDER, exist_ok=True)
CSV_PATH = os.path.join(DATA_FOLDER, "XRD_final_results.csv")

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV file missing: {CSV_PATH}\nRun upstream fitting code first to generate this csv!")

D_MAX_THRESHOLD = 600  # Filter abnormal large grain size

# Color definition
COLOR_FIT = {'β': '#2ecc71', 'κ': '#3498db'}
COLOR_PHASE = {'α': '#e74c3c', 'β': '#2ecc71', 'κ': '#3498db'}
MARKERS = {'Thin film (~100 nm)': 'o', 'Thick film (~400 nm)': 's'}
THIN_TAG = {'Thin film (~100 nm)': 'thin', 'Thick film (~400 nm)': 'thick'}

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

def save_fig(filename):
    """Save PNG (1200 dpi) and vector PDF into DATA folder"""
    out_png = os.path.join(DATA_FOLDER, f"{filename}.png")
    out_pdf = os.path.join(DATA_FOLDER, f"{filename}.pdf")
    plt.savefig(out_png, dpi=1200, bbox_inches='tight')
    plt.savefig(out_pdf, bbox_inches='tight')
    plt.close()

# ============================================================
# Fig1: Grain size - microstrain linear fitting (β/κ phase only)
# ============================================================
def plot_grain_strain(df):
    df_f = df.copy()
    for col in ['β-phase grain size (nm)', 'κ-phase grain size (nm)']:
        df_f.loc[df_f[col] > D_MAX_THRESHOLD, col] = np.nan
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for ph, dc, ec in [
        ('β', 'β-phase grain size (nm)', 'β-phase microstrain'),
        ('κ', 'κ-phase grain size (nm)', 'κ-phase microstrain'),
    ]:
        data = df_f[df_f[dc].notna() & df_f[ec].notna()]
        x_all, y_all = data[dc], data[ec]
        # Scatter: different markers for film thickness groups
        for group in ['Thin film (~100 nm)', 'Thick film (~400 nm)']:
            g = data[data['Thickness groups'] == group]
            if not g.empty:
                ax.scatter(g[dc], g[ec],
                           c=COLOR_FIT[ph], marker=MARKERS[group],
                           s=55, alpha=0.85, edgecolors='black', linewidths=0.7,
                           label=f"{ph}-phase ({THIN_TAG[group]})")
        # Linear fitting + Pearson correlation coefficient
        if len(x_all) >= 2:
            z = np.polyfit(x_all, y_all, 1)
            xf = np.linspace(x_all.min(), x_all.max(), 100)
            yf = z[0] * xf + z[1]
            corr, _ = pearsonr(x_all, y_all)
            ax.plot(xf, yf, c=COLOR_FIT[ph], lw=1.8, ls='-',
                    label=f"{ph}-phase fit, r = {corr:.2f}")
    ax.set_xlabel("Grain size (nm)")
    ax.set_ylabel("Microstrain")
    ax.legend(loc='upper right', frameon=False, ncol=1, bbox_to_anchor=(0.98, 1.02), borderaxespad=0)
    ax.grid(alpha=0.3, linestyle='--')
    save_fig("Fig1_grain_strain_linear_fit")

# ============================================================
# Fig2: Phase fraction under different growth conditions (thin/thick film comparison, dashed connecting arrow)
# ============================================================
def plot_phase_transition(df):
    # Only compare samples without Si doping (Si=0)
    df = df[df['Si(μmol/min)'] == 0].copy()
    # Group keys for growth conditions: (P, Ar), Si fixed at 0
    group_cols = ['P(mbar)', 'Ar(sccm)']
    conditions = (df.groupby(group_cols).size()
                  .reset_index().sort_values(group_cols)
                  .reset_index(drop=True))
    n_cond = len(conditions)
    # Color scheme: α red / β green / κ blue, consistent with Fig1
    COLOR_PHASE = {'α': '#e74c3c', 'β': '#2ecc71', 'κ': '#3498db'}
    # Wider canvas to reserve space for external legend on right
    fig, ax = plt.subplots(figsize=(8.8, 4.5))
    frac_cols = {'α': 'α-phase fraction',
                 'β': 'β-phase fraction',
                 'κ': 'κ-phase fraction'}
    x_pos = np.arange(n_cond)
    offsets = {'α': -0.20, 'β': 0.0, 'κ': 0.20}
    # ---- Scatter points ----
    for ph in ['α', 'β', 'κ']:
        col = frac_cols[ph]
        for group in ['Thin film (~100 nm)', 'Thick film (~400 nm)']:
            for i, cond in conditions.iterrows():
                mask = ((df['P(mbar)'] == cond['P(mbar)']) &
                        (df['Ar(sccm)'] == cond['Ar(sccm)']) &
                        (df['Thickness groups'] == group))
                sub = df[mask]
                if not sub.empty:
                    val = sub[col].mean() * 100
                    ax.scatter(x_pos[i] + offsets[ph], val,
                               c=COLOR_PHASE[ph], marker=MARKERS[group],
                               s=55, alpha=0.9, edgecolors='black', linewidths=0.7,
                               label=(f"{ph}-phase ({THIN_TAG[group]})" if i == 0 else ""))
    # ---- Arrow for same phase: thin film → thick film under identical growth condition ----
    for ph in ['α', 'β', 'κ']:
        col = frac_cols[ph]
        for i, cond in conditions.iterrows():
            y_thin = np.nan
            y_thick = np.nan
            # Get y value for thin and thick film
            mask_thin = ((df['P(mbar)'] == cond['P(mbar)']) &
                         (df['Ar(sccm)'] == cond['Ar(sccm)']) &
                         (df['Thickness groups'] == 'Thin film (~100 nm)'))
            sub_thin = df[mask_thin]
            if not sub_thin.empty:
                y_thin = sub_thin[col].mean() * 100
            mask_thick = ((df['P(mbar)'] == cond['P(mbar)']) &
                          (df['Ar(sccm)'] == cond['Ar(sccm)']) &
                          (df['Thickness groups'] == 'Thick film (~400 nm)'))
            sub_thick = df[mask_thick]
            if not sub_thick.empty:
                y_thick = sub_thick[col].mean() * 100
            if not (np.isnan(y_thin) or np.isnan(y_thick)):
                x_val = x_pos[i] + offsets[ph]
                ax.annotate("",
                            xy=(x_val, y_thick),        # Arrow end: thick film
                            xytext=(x_val, y_thin),     # Arrow start: thin film
                            arrowprops=dict(arrowstyle="->",
                                            color=COLOR_PHASE[ph],
                                            lw=1.0,
                                            linestyle='--',
                                            alpha=0.55))
    ax.set_xticks(x_pos)
    xtick_text = []
    for i, c in conditions.iterrows():
        txt = f"Condition {i+1}\nP={int(c['P(mbar)'])}, Ar={int(c['Ar(sccm)'])}"
        xtick_text.append(txt)
    ax.set_xticklabels(xtick_text)
    ax.set_xlabel("Growth condition")
    ax.set_ylabel("Phase fraction (%)")
    ax.set_ylim(-5, 105)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0),
              frameon=False, ncol=1, fontsize=16)
    ax.grid(alpha=0.3, linestyle='--', axis='y')
    plt.tight_layout()
    save_fig("Fig2_phase_fraction_by_condition")

# Execute plotting
if __name__ == "__main__":
    plot_grain_strain(df)
    print("Fig1 (grain-strain linear fit) saved to DATA folder")
    plot_phase_transition(df)
    print("Fig2 (phase fraction by condition) saved to DATA folder")
    print("Both figures generated (PNG 1200dpi + PDF).")


Fig1 (grain-strain linear fit) saved to DATA folder
Fig2 (phase fraction by condition) saved to DATA folder
Both figures generated (PNG 1200dpi + PDF).


In [17]:
# ============================================================
# Extract data from XRD_final_results.csv for machine learning progressive analysis
# 2 figures for each group:
#   (1) RF feature importance + |r| + R² grouped bar chart
#   (2) PLS(1D) -> GPR fitting with 95% confidence interval
# Output: PNG (1200 dpi) + vector PDF saved under DATA/ML_analysis/
# ============================================================

# ---------- Journal style configuration (match Fig3 font size & bold weight) ----------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 0.8

# ---------- File path (consistent with all plotting scripts) ----------
DATA_FOLDER = "DATA"
CSV_PATH = os.path.join(DATA_FOLDER, "XRD_final_results.csv")
ML_FOLDER = os.path.join(DATA_FOLDER, "ML_analysis")
os.makedirs(ML_FOLDER, exist_ok=True)

# ---------- Feature column mapping, full name -> short label ----------
FEATURE_COLS = ['Ar(sccm)', 'P(mbar)', 'Si(μmol/min)', 'Epitaxial growth time(s)']
FEATURE_SHORT = {'Ar(sccm)': 'Ar',
                 'P(mbar)': 'P',
                 'Si(μmol/min)': 'Si',
                 'Epitaxial growth time(s)': 'Time'}

ALPHA_COL = 'α-phase fraction'
BETA_D_COL = 'β-phase grain size (nm)'
KAPPA_D_COL = 'κ-phase grain size (nm)'

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

def save_fig(filename):
    """Save PNG (1200 dpi) and vector PDF to DATA/ML_analysis"""
    png_path = os.path.join(ML_FOLDER, f"{filename}.png")
    pdf_path = os.path.join(ML_FOLDER, f"{filename}.pdf")
    plt.savefig(png_path, dpi=1200, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.close()

# ============================================================
# Calculate RF feature importance
# ============================================================
def calc_rf_importance(X_scaled, y):
    rf = RandomForestRegressor(n_estimators=50, random_state=42)
    rf.fit(X_scaled, y)
    return dict(zip(FEATURE_COLS, np.round(rf.feature_importances_, 3)))

def calc_corr_and_r2(X, y):
    corr_dict, r2_dict = {}, {}
    for i, feat in enumerate(FEATURE_COLS):
        corr = np.abs(np.corrcoef(X[:, i], y)[0, 1])
        corr_dict[feat] = round(corr, 3)
        lr = LinearRegression()
        lr.fit(X[:, [i]], y)
        r2_dict[feat] = round(lr.score(X[:, [i]], y), 3)
    return corr_dict, r2_dict

# ============================================================
# Fig(1): Grouped bar chart of RF importance, |r| and R²
# ============================================================
def plot_importance(desc, file_name, corr_dict, r2_dict, rf_imp):
    sorted_feats = sorted(FEATURE_COLS, key=lambda f: rf_imp[f], reverse=True)
    x = np.arange(len(sorted_feats))
    width = 0.25
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    ax.bar(x - width, [corr_dict[f] for f in sorted_feats], width,
           label='|r|', color='#2ecc71')
    ax.bar(x, [r2_dict[f] for f in sorted_feats], width,
           label=r'R$^2$', color='#3498db')
    ax.bar(x + width, [rf_imp[f] for f in sorted_feats], width,
           label='RF importance', color='#e74c3c')
    ax.set_xticks(x)
    ax.set_xticklabels([FEATURE_SHORT[f] for f in sorted_feats])
    ax.set_xlabel("Process parameter")
    ax.set_ylabel("Importance")
    ax.set_title(desc)
    ax.legend(loc='upper right', frameon=False, ncol=1, bbox_to_anchor=(1.018, 1.02), borderaxespad=0)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    save_fig(f"{file_name}_importance")

# ============================================================
# Fig(2): PLS(1D) -> GPR fitting with 95% confidence interval
# ============================================================
def plot_pls_gpr(desc, file_name, X_scaled, y):
    pls = PLSRegression(n_components=1)
    X_pls = pls.fit_transform(X_scaled, y)[0]
    X_pls_1d = X_pls.flatten()
    gpr = GaussianProcessRegressor(
        kernel=C(1.0) * Matern(0.5),
        n_restarts_optimizer=10, random_state=42)
    gpr.fit(X_pls, y)
    X_plot = np.linspace(X_pls_1d.min() * 0.9, X_pls_1d.max() * 1.1, 200).reshape(-1, 1)
    y_pred, sigma = gpr.predict(X_plot, return_std=True)

    fig, ax = plt.subplots(figsize=(7.5, 4.0))
    ax.fill_between(X_plot.flatten(),
                    y_pred - 1.96 * sigma, y_pred + 1.96 * sigma,
                    alpha=0.10, color='red', label='95% CI')
    ax.plot(X_plot, y_pred, 'r-', lw=2.0, label='GPR fit')
    ax.scatter(X_pls_1d, y, c='#3498db', s=60, edgecolors='black',
               linewidths=0.7, zorder=5, label='Experimental')
    ax.set_xlabel("PLS component (supersaturation)")
    ax.set_ylabel("Grain size (nm)")
    ax.set_title(desc)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0),
              frameon=False, ncol=1)
    ax.grid(linestyle='--', alpha=0.4)
    plt.tight_layout()
    save_fig(f"{file_name}_GPR")

# ================== run analysis: NaN filtering & sample size protection ==================
def run_analysis(df_sub, target_col, title_desc, filename_prefix):
    df_sub = df_sub.dropna(subset=[target_col])
    if len(df_sub) < 3:
        print(f"WARNING: {filename_prefix}: Only {len(df_sub)} valid samples, skip plotting and training.")
        return
    X = df_sub[FEATURE_COLS].values
    y = df_sub[target_col].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    rf_imp = calc_rf_importance(X_scaled, y)
    corr_dict, r2_dict = calc_corr_and_r2(X, y)
    plot_importance(title_desc, filename_prefix, corr_dict, r2_dict, rf_imp)
    plot_pls_gpr(title_desc, filename_prefix, X_scaled, y)

# ============================================================
# Main routine: Five progressive analysis groups
# ============================================================
if __name__ == "__main__":
    print(f"Loaded {len(df)} samples from {CSV_PATH}\n")
    run_analysis(df, BETA_D_COL,
                 "β-phase growth kinetics", "01_beta_reference")
    run_analysis(df, KAPPA_D_COL,
                 "κ-phase (all samples)", "02_kappa_all")
    df_S2 = df[df[ALPHA_COL] < 0.05]
    run_analysis(df[df[ALPHA_COL] <= 0.25], KAPPA_D_COL,
                 "κ-phase (α ≤ 25%)", "03_kappa_alpha_le_25")
    run_analysis(df_S2, KAPPA_D_COL,
                 "κ-phase (α < 5%)", "04_kappa_alpha_lt_5")
    sample13 = df[df['Sample'] == 13]
    df_S4 = pd.concat([df_S2, sample13], ignore_index=True).drop_duplicates()
    run_analysis(df_S4, KAPPA_D_COL,
                 "κ-phase (α < 5% + Sample 13)", "05_kappa_alpha_lt_5_add13")
    print(f"\nAll ML figures saved to: {ML_FOLDER}/")


Loaded 16 samples from DATA\XRD_final_results.csv


All ML figures saved to: DATA\ML_analysis/


In [18]:
# =============================================================================
# Plot phase fraction scatter plot with arrows for phase transition visualization
# Data source: XRD_final_results.csv stored in ./DATA
# Output: PNG and vector PDF figures saved to ./DATA
# Consistent font size & bold style aligned with Figure 3 for journal submission
# =============================================================================
warnings.filterwarnings("ignore")

# -------------------------- Journal Figure Style Settings --------------------------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 1200
plt.rcParams['axes.linewidth'] = 0.8

# Font size & bold setting, fully consistent with Fig.3
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# -------------------------- Relative File Path Configuration --------------------------
DATA_FOLDER = "DATA"
csv_path = os.path.join(DATA_FOLDER, "XRD_final_results.csv")
save_png = os.path.join(DATA_FOLDER, "phase_fraction_correct_ratio.png")
save_pdf = os.path.join(DATA_FOLDER, "phase_fraction_correct_ratio.pdf")

# Load experimental dataset
df = pd.read_csv(csv_path, encoding="utf-8-sig")

# Definition of four growth conditions
cond_def = [
    {"Ar":40, "P":20, "base_x":0, "label":"Condition 1\nP=20, Ar=40"},
    {"Ar":60, "P":20, "base_x":1, "label":"Condition 2\nP=20, Ar=60"},
    {"Ar":80, "P":20, "base_x":2, "label":"Condition 3\nP=20, Ar=80"},
    {"Ar":80, "P":40, "base_x":3, "label":"Condition 4\nP=40, Ar=80"},
]

# Six sample categories within each growth condition
cat_list = [
    {"key":"α_thin",  "phase":"α", "thick":"Thin",  "marker":"o", "color":"#d62728"},
    {"key":"α_thick", "phase":"α", "thick":"Thick", "marker":"s", "color":"#d62728"},
    {"key":"κ_thin",  "phase":"κ", "thick":"Thin",  "marker":"o", "color":"#1f77b4"},
    {"key":"κ_thick", "phase":"κ", "thick":"Thick", "marker":"s", "color":"#1f77b4"},
    {"key":"β_thin",  "phase":"β", "thick":"Thin",  "marker":"o", "color":"#2ca02c"},
    {"key":"β_thick", "phase":"β", "thick":"Thick", "marker":"s", "color":"#2ca02c"},
]

col_spacing = 0.13
phase_col_map = {
    "α": "α-phase fraction",
    "κ": "κ-phase fraction",
    "β": "β-phase fraction",
}

# Canvas dimension: aspect ratio ~2.31, compatible with journal single-column / wide figure
fig, ax = plt.subplots(figsize=(20.8, 9))
y_min, y_max = -6, 106

for cond in cond_def:
    ar_val = cond["Ar"]
    p_val = cond["P"]
    base_x = cond["base_x"]
    # Filter data by Ar flow and chamber pressure
    df_cond = df[(df["Ar(sccm)"] == ar_val) & (df["P(mbar)"] == p_val)].copy()

    for cat_idx, cat in enumerate(cat_list):
        cat_x = base_x + (cat_idx - 2.5) * col_spacing
        ph = cat["phase"]
        thick_type = cat["thick"]
        mk = cat["marker"]
        c = cat["color"]
        # Filter thin/thick samples
        df_sub = df_cond[df_cond["Thickness groups"].str.contains(thick_type, na=False)]
        if len(df_sub) == 0:
            continue

        all_points = []
        # Undoped reference point (Si flux = 0)
        row_0 = df_sub[df_sub["Si(μmol/min)"] == 0]
        if len(row_0) == 0:
            continue
        y0 = row_0[phase_col_map[ph]].iloc[0] * 100
        x0 = cat_x - 0.025
        all_points.append({
            "y": y0, "x": x0,
            "is_doped": False, "row": row_0.iloc[0]
        })

        # Doped samples with Si incorporation
        doped_rows = df_sub[df_sub["Si(μmol/min)"] > 0].reset_index(drop=True)
        for d_idx, (_, row_d) in enumerate(doped_rows.iterrows()):
            yd = row_d[phase_col_map[ph]] * 100
            xd = cat_x + 0.025
            all_points.append({
                "y": yd, "x": xd,
                "is_doped": True, "row": row_d
            })

        # Sort points by phase fraction value
        all_points.sort(key=lambda p: p["y"])

        # Plot undoped reference marker (hollow symbol)
        ax.scatter(x0, y0, marker=mk, facecolors="white",
                   edgecolors=c, s=90, linewidth=2, zorder=4)

        # Plot doped markers and transition arrows
        for p_idx, p in enumerate(all_points):
            if not p["is_doped"]:
                continue
            yd = p["y"]
            xd = p["x"]
            ax.scatter(xd, yd, marker=mk, facecolors=c,
                       edgecolors=c, s=90, linewidth=2, zorder=4)
            # Dashed arrow to visualize phase evolution upon Si doping
            arrow = FancyArrowPatch((x0, y0), (xd, yd),
                                    arrowstyle="->", color=c,
                                    linestyle="--", linewidth=1.2,
                                    mutation_scale=15, alpha=0.9, zorder=2)
            ax.add_patch(arrow)

# Axes configuration
ax.set_xticks([c["base_x"] for c in cond_def])
ax.set_xticklabels([c["label"] for c in cond_def])
ax.set_ylabel("Phase fraction (%)")
ax.set_xlabel("Growth condition")
ax.set_ylim(y_min, y_max)
ax.grid(axis="y", linestyle="--", alpha=0.45)

# Custom legend definition
legend_elements = [
    Line2D([],[],marker="o",mfc="white",mec="#d62728",linestyle="",markersize=9,label=r'$\alpha$-phase (thin)'),
    Line2D([],[],marker="s",mfc="white",mec="#d62728",linestyle="",markersize=9,label=r'$\alpha$-phase (thick)'),
    Line2D([],[],marker="o",mfc="white",mec="#1f77b4",linestyle="",markersize=9,label=r'$\kappa$-phase (thin)'),
    Line2D([],[],marker="s",mfc="white",mec="#1f77b4",linestyle="",markersize=9,label=r'$\kappa$-phase (thick)'),
    Line2D([],[],marker="o",mfc="white",mec="#2ca02c",linestyle="",markersize=9,label=r'$\beta$-phase (thin)'),
    Line2D([],[],marker="s",mfc="white",mec="#2ca02c",linestyle="",markersize=9,label=r'$\beta$-phase (thick)'),
    Line2D([],[],marker="o",mfc="white",mec="black",linestyle="",markersize=9,label="Undoped"),
    Line2D([],[],marker="o",mfc="black",mec="black",linestyle="",markersize=9,label="Doped"),
]
ax.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=False, labelspacing=0.8)

plt.tight_layout()
# Save figure
plt.savefig(save_png, bbox_inches="tight", facecolor="white")
plt.savefig(save_pdf, bbox_inches="tight", facecolor="white")
plt.close()

print(f"✅ Plot finished. Annotations of ε and Si values removed.")
print(f"Font size & bold style matched with Fig.3.")
print(f"Figure size: width 20.8 × height 9, aspect ratio ≈2.31")
print(f"Output files:")
print(f"  {save_png}")
print(f"  {save_pdf}")


✅ Plot finished. Annotations of ε and Si values removed.
Font size & bold style matched with Fig.3.
Figure size: width 20.8 × height 9, aspect ratio ≈2.31
Output files:
  DATA\phase_fraction_correct_ratio.png
  DATA\phase_fraction_correct_ratio.pdf
